# Cuadernillo 1 · Mirar los datos

**Autor:** Dilan Aldana - Jeison Del Rio

**Objetivo.** Conocer el archivo antes de analizarlo: cuántas solicitudes hay, qué trae cada columna, qué datos faltan y cómo se reparten las respuestas.

**Fecha:** 22 de noviembre de 2026

**Desarrollo.** Cargar el archivo, revisar los valores faltantes y contar aprobados y rechazados. Al final, una **conclusión** con lo que estos hallazgos implican para los cuadernillos siguientes.

In [1]:

# Python básico
import sys
import warnings
from pathlib import Path

# Datos
import pandas as pd
from IPython.display import display

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(RAIZ))

from funciones import analisis

warnings.filterwarnings("ignore")
analisis.configurar_estilo()

print("Todo listo")


Todo listo


**Qué hace esta celda.** Deja listas las herramientas: `pandas` para manejar tablas, `analisis` con las funciones del proyecto y `Path` para ubicar la carpeta. No calcula nada todavía.

## 1. Cargar el archivo

El banco entrega los datos en un **CSV**, un archivo de texto donde cada línea es una solicitud y las columnas se separan con comas. `pandas` lo convierte en una tabla (`df`).

In [2]:
df = pd.read_csv(RAIZ / "datos" / "solicitudes_prestamos.csv")

print(f"Filas: {len(df)}   Columnas: {df.shape[1]}")
df.head(5)

Filas: 614   Columnas: 13


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


La tabla tiene **614 solicitudes** (filas) y **13 columnas**. Cada fila es un caso y cada columna un dato de ese caso: ingresos y monto del préstamo (números), sexo, estado civil, educación, empleo y zona (texto), `Loan_ID` (un código único, como un número de radicado) y `Loan_Status` (la respuesta: `Y` aprobado, `N` rechazado), que es lo que el modelo debe predecir.

## 2. Valores faltantes

Un **nulo** es una casilla que el banco no registró. La celda cuenta cuántos hay en cada columna.

In [3]:
print("Valores faltantes por columna:")
display(analisis.resumen_nulos(df))

Valores faltantes por columna:


,nulos,porcentaje
Credit_History,50,8.1
Self_Employed,32,5.2
LoanAmount,22,3.6
Dependents,15,2.4
Loan_Amount_Term,14,2.3
Gender,13,2.1
Married,3,0.5


**7 de las 13 columnas tienen huecos.** La más incompleta es `Credit_History` (50), seguida de `Self_Employed` (32), `LoanAmount` (22) y `Dependents` (15).

Borrarlas costaría **134 solicitudes (21,8 %)**, casi una de cada cinco. Por eso no se eliminan: los huecos se **imputan** (se rellenan con la mediana) dentro del modelo (cuadernillos 3 y 4).

## 3. Duplicados y variable objetivo

Se revisa si hay solicitudes repetidas y cómo se reparten las respuestas. `Loan_Status` es la **variable objetivo**.

In [4]:
print("¿Cuántas solicitudes se aprobaron?")
print(df["Loan_Status"].value_counts())
print(f"\nPorcentaje aprobado: {df['Loan_Status'].eq('Y').mean() * 100:.1f}%")
print("Filas duplicadas:", df.duplicated().sum())

¿Cuántas solicitudes se aprobaron?
Loan_Status
Y    422
N    192
Name: count, dtype: int64

Porcentaje aprobado: 68.7%
Filas duplicadas: 0


Hay **422 aprobadas (68,7 %)** y **192 rechazadas (31,3 %)**, y ninguna fila repetida.

Esa diferencia se llama **desbalanceo**. Si el modelo dijera siempre "aprobado", acertaría el 68,7 % sin haber aprendido nada y no detectaría ningún rechazo. Para evitarlo, al entrenar se le da más peso a la clase minoritaria con `class_weight="balanced"` (cuadernillo 4).

## 4. Conclusión

- El archivo tiene **614 solicitudes y 13 columnas**; `Loan_ID` es solo un código y se quitará (cuadernillo 3).
- **7 columnas tienen huecos** y 134 filas (21,8 %) tienen al menos uno. Se imputan con la mediana en vez de borrarlas (cuadernillos 3 y 4).
- Las clases están **desbalanceadas (68,7 % / 31,3 %)** y se compensan con `class_weight="balanced"` (cuadernillo 4).

Los datos sirven, pero hay que limpiarlos y tratarlos antes de entrenar.